# Statistics That Matter for Data Science

Practical statistical concepts that appear in every DS project:
hypothesis testing, the bootstrap, power analysis, multiple comparisons.

## Hypothesis testing and p-values

In [ ]:
import numpy as np
from scipy import stats

rng = np.random.default_rng(42)

# Simulate A/B test: does the treatment group have higher conversion?
control   = rng.binomial(1, 0.10, 1000)   # 10% conversion
treatment = rng.binomial(1, 0.12, 1000)   # 12% conversion (true lift = +2pp)

t_stat, p_val = stats.ttest_ind(control, treatment)
print(f"Control mean:   {control.mean():.3f}")
print(f"Treatment mean: {treatment.mean():.3f}")
print(f"Observed diff:  {treatment.mean() - control.mean():+.3f}")
print(f"t-statistic:    {t_stat:.3f}")
print(f"p-value:        {p_val:.4f}  {'(significant at α=0.05)' if p_val < 0.05 else '(not significant)'}")

## Bootstrap confidence intervals

Model-free uncertainty estimate — no distributional assumptions required.

In [ ]:
import numpy as np

rng = np.random.default_rng(42)
data = rng.lognormal(mean=3, sigma=0.5, size=200)  # skewed revenue data

# Bootstrap 95% CI for the mean
n_boot = 5000
boot_means = [rng.choice(data, size=len(data), replace=True).mean()
              for _ in range(n_boot)]
ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])
print(f"Sample mean:        {data.mean():.2f}")
print(f"Bootstrap 95% CI:   [{ci_low:.2f}, {ci_high:.2f}]")
print(f"Parametric 95% CI:  [{data.mean() - 1.96*data.std()/len(data)**0.5:.2f}, "
      f"{data.mean() + 1.96*data.std()/len(data)**0.5:.2f}]")
print("Bootstrap CI is wider — correctly accounts for skew")

## Power analysis — how many samples do I need?

In [ ]:
from statsmodels.stats.power import TTestIndPower

analysis = TTestIndPower()

# Scenario: detect a 2pp lift in conversion rate (baseline 10%)
# Baseline std ≈ sqrt(0.10 * 0.90)
baseline_std = (0.10 * 0.90) ** 0.5
effect_size = 0.02 / baseline_std   # Cohen's d

n = analysis.solve_power(effect_size=effect_size, alpha=0.05, power=0.80)
print(f"Effect size (Cohen's d): {effect_size:.3f}")
print(f"Required n per arm:      {int(n) + 1}")
print(f"Total sample required:   {2*(int(n)+1)}")
print()
# Sensitivity: how does n change with power requirement?
for power in [0.70, 0.80, 0.90, 0.95]:
    n = analysis.solve_power(effect_size=effect_size, alpha=0.05, power=power)
    print(f"  Power={power:.0%} → n per arm = {int(n)+1}")

## Multiple comparisons — Benjamini-Hochberg correction

In [ ]:
import numpy as np
from statsmodels.stats.multitest import multipletests

rng = np.random.default_rng(42)

# Simulate 20 hypothesis tests; only 3 truly significant
p_values = np.concatenate([
    rng.uniform(0, 0.01, 3),   # truly significant
    rng.uniform(0.05, 1.0, 17) # noise
])
rng.shuffle(p_values)

reject_bonf, _, _, _ = multipletests(p_values, alpha=0.05, method='bonferroni')
reject_bh,   _, _, _ = multipletests(p_values, alpha=0.05, method='fdr_bh')

print(f"Raw p < 0.05:      {(p_values < 0.05).sum()} rejections (likely includes false positives)")
print(f"Bonferroni:        {reject_bonf.sum()} rejections (very conservative)")
print(f"Benjamini-Hochberg:{reject_bh.sum()} rejections (controls FDR at 5%)")
print()
print("BH is the standard for DS/ML multiple-testing scenarios")